In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Ignoriamo i warning per i geni troppo rari che non riescono a convergere nel modello
warnings.filterwarnings('ignore')

# ==========================================
# 1. APPROCCIO CLASSICO (Sbagliato per popolazioni miste)
# Usa il Test Esatto di Fisher, ignorando la razza.
# ==========================================

def formatta_dati_classico(csv_path):
    """
    Legge il CSV e rimuove la colonna 'Razza' perché l'approccio classico non la usa.
    """
    df = pd.read_csv(csv_path)
    if 'Patient_ID' in df.columns:
        df = df.set_index('Patient_ID')
    if 'Razza' in df.columns:
        df = df.drop(columns=['Razza']) # Ignoriamo il fattore etnico
    return df

def costruisci_mappa_classica(df, target='RAS', soglia_p=0.05):
    """
    Calcola l'Odds Ratio (probabilità di mutare insieme) usando il Test di Fisher.
    """
    risultati = []
    geni = [g for g in df.columns if g != target]
    
    for gene in geni:
        # Crea una tabella di contingenza (quante volte mutano insieme, separati, o nessuno dei due)
        tabella = pd.crosstab(df[target], df[gene])
        
        # Se la tabella è 2x2 (ci sono sia 0 che 1 per entrambi i geni) procediamo
        if tabella.shape == (2, 2):
            odds_ratio, p_value = stats.fisher_exact(tabella)
            risultati.append({
                'Gene': gene,
                'Odds_Ratio': odds_ratio,
                'P_Value': p_value
            })
            
    df_risultati = pd.DataFrame(risultati)
    # Filtriamo solo i risultati statisticamente significativi
    df_significativi = df_risultati[df_risultati['P_Value'] < soglia_p].copy()
    
    plotta_risultati(df_significativi, "Mappa Classica (Fisher) - Ignora la Razza")
    return df_risultati


# ==========================================
# 2. APPROCCIO CORRETTO (Regressione Logistica)
# Usa la razza come covariata per eliminare i falsi positivi.
# ==========================================

def formatta_dati_corretto(csv_path):
    """
    Legge il CSV e mantiene la colonna 'Razza'.
    """
    df = pd.read_csv(csv_path)
    if 'Patient_ID' in df.columns:
        df = df.set_index('Patient_ID')
    
    # Assicuriamoci che i nomi delle colonne non abbiano spazi o caratteri strani 
    # che darebbero fastidio alla formula della regressione
    df.columns = df.columns.str.replace(' ', '_').str.replace('-', '_')
    return df

def costruisci_mappa_corretta(df, target='RAS', col_razza='Razza', soglia_p=0.05):
    """
    Calcola l'Odds Ratio "Aggiustato" usando la regressione logistica multivariata.
    """
    risultati = []
    geni = [g for g in df.columns if g not in [target, col_razza]]
    
    for gene in geni:
        # Saltiamo i geni troppo rari (mutati in meno del 3% dei pazienti) per evitare crash matematici
        if df[gene].mean() < 0.03:
            continue
            
        # Formula: Cerca di prevedere la mutazione del Gene in base a RAS e alla Razza
        # C() dice al modello che la Razza è una categoria (testo), non un numero
        formula = f"{gene} ~ {target} + C({col_razza})"
        
        try:
            # Fit del modello logistico
            modello = smf.logit(formula=formula, data=df).fit(disp=0)
            
            # Estraiamo i valori per RAS
            p_value = modello.pvalues[target]
            coeff = modello.params[target]
            odds_ratio_aggiustato = np.exp(coeff) # La matematica richiede l'esponenziale del coefficiente
            
            risultati.append({
                'Gene': gene,
                'Odds_Ratio': odds_ratio_aggiustato,
                'P_Value': p_value
            })
        except Exception:
            # Il modello fallisce se un gene è mutato solo in una razza (separazione perfetta)
            pass
            
    df_risultati = pd.DataFrame(risultati)
    df_significativi = df_risultati[df_risultati['P_Value'] < soglia_p].copy()
    
    plotta_risultati(df_significativi, "Mappa Corretta (Regressione Logistica) - Aggiustata per Razza")
    return df_risultati


# ==========================================
# 3. FUNZIONE DI PLOTTING (Grafica)
# ==========================================

def plotta_risultati(df_plot, titolo):
    """
    Crea un grafico a barre (Forest-like plot) per mostrare l'intensità della co-occorrenza.
    """
    if df_plot.empty:
        print(f"[{titolo}]: Nessuna co-occorrenza significativa trovata.")
        return

    # Sostituiamo valori "infiniti" di Odds Ratio (causati da divisioni per zero) con un numero alto
    df_plot['Odds_Ratio'] = df_plot['Odds_Ratio'].replace([np.inf, -np.inf], 100)
    
    # Ordiniamo i geni dal più forte al più debole
    df_plot = df_plot.sort_values(by='Odds_Ratio', ascending=False)
    
    plt.figure(figsize=(10, 6))
    
    # Creiamo un grafico a barre orizzontali. 
    # Usiamo il logaritmo in base 2 dell'Odds Ratio per centrare il grafico sullo zero.
    # Valori > 0 indicano Co-occorrenza (mutano insieme). Valori < 0 indicano Mutua Esclusività (si evitano).
    sns.barplot(
        x=np.log2(df_plot['Odds_Ratio']), 
        y=df_plot['Gene'], 
        palette="vlag"
    )
    
    plt.axvline(0, color='black', linewidth=1, linestyle='--')
    plt.title(titolo, fontsize=14, fontweight='bold')
    plt.xlabel('Log2(Odds Ratio)\n<-- Si evitano | Mutano Insieme -->', fontsize=12)
    plt.ylabel('Geni', fontsize=12)
    
    plt.tight_layout()
    plt.show()

# ==========================================
# ESEMPIO DI ESECUZIONE 
# (Sostituisci 'tuo_file.csv' con il nome reale del tuo file)
# ==========================================
if __name__ == "__main__":
    # csv_path = "dati_pazienti.csv"
    
    # print("Esecuzione Modello Classico...")
    # df_classico = formatta_dati_classico(csv_path)
    # res_classico = costruisci_mappa_classica(df_classico, target='RAS')
    
    # print("\nEsecuzione Modello Corretto...")
    # df_corretto = formatta_dati_corretto(csv_path)
    # res_corretto = costruisci_mappa_corretta(df_corretto, target='RAS', col_razza='Razza')